### Install Libraries

### Import Libraries

In [1]:
# from langchain_community.llms import CTransformers
# from langchain_community.llms import LlamaCpp # <- llamaCpp! An Alternate option for CTransformers - Make a Poll.
# from langchain.callbacks.manager import CallbackManager
from langchain.callbacks.streaming_stdout import StreamingStdOutCallbackHandler
from langchain.chains import LLMChain
from langchain.prompts import PromptTemplate

from langchain.chains import ConversationChain
from langchain.chains.conversation.memory import (ConversationBufferMemory, 
                                                  ConversationSummaryMemory, 
                                                  ConversationBufferWindowMemory,
                                                  ConversationKGMemory)

In [2]:
# RAG 1st
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.document_loaders import TextLoader
from langchain_community.embeddings.sentence_transformer import (
    SentenceTransformerEmbeddings,
)

from langchain.storage import InMemoryStore
from langchain_community.document_loaders import TextLoader

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.retrievers import ParentDocumentRetriever
from langchain_community.vectorstores import Chroma
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter

### Retriever

In [3]:
loader = PyMuPDFLoader(".\\Data\\PDFs\\DepressionGuide-web.pdf")
documents  = loader.load()

In [4]:
embedding_function = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")

c:\Users\User\anaconda3\envs\omdena_chatbot\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=2000)

child_splitter = RecursiveCharacterTextSplitter(chunk_size=400)

vectorstore = Chroma(collection_name="split_parents", embedding_function=embedding_function)

store = InMemoryStore()

c:\Users\User\anaconda3\envs\omdena_chatbot\Lib\site-packages\onnxruntime\capi\onnxruntime_validation.py:26: UserWarning: Unsupported Windows version (11). ONNX Runtime supports Windows 10 and above, only.
  warnings.warn(


In [6]:
retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)

In [7]:
retriever.add_documents(documents)

In [8]:
# Testing
retriever.get_relevant_documents("I'm Tired all the time, feeling “lazy”")

c:\Users\User\anaconda3\envs\omdena_chatbot\Lib\site-packages\langchain_core\_api\deprecation.py:119: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 0.3.0. Use invoke instead.
  warn_deprecated(


[Document(page_content='Depression: Parents’ Medication Guide       5\nCauses and Symptoms\nWhy does my child \nhave depression?\nWe don’t fully understand all the \ncauses of depression; we think it’s a \ncombination of genetics (inherited traits) \nand environmental factors (events and \nsurroundings). There is no single cause. \nStressors or events that cause a stressful \nresponse and genetic factors can cause \ndepression. Stressors can be triggers \nthat result from pediatric illnesses and \ndiseases, such as viral infections; diseases \nof the thyroid and endocrine system; head \ninjury; epilepsy; and heart, kidney, and lung \ndiseases. A family history of depression \nis a major genetic factor; a child can be \nmore prone to becoming depressed if \na parent or sibling has been diagnosed \nwith depression. Stressors in everyday \nlife also contribute to the development \nof depression, for example, the loss of a \nclose loved one; parents frequently arguing, \nseparating, or div

### Augmentor

In [12]:
from langchain import PromptTemplate
from langchain.prompts.chat import (
    ChatPromptTemplate,
    SystemMessagePromptTemplate,
    AIMessagePromptTemplate,
    HumanMessagePromptTemplate,
)

# Define system and user message templates
system_message_template = '''You are a Mental Health Specialist (therapist).
Your job is to provide support for individuals with Depressive Disorder.
Act as a compassionate listener and offer helpful responses based on the user's queries.
If the user seeks casual conversation, be friendly and supportive.
If they seek factual information, use the context of the conversation to provide relevant responses.
If unsure, be honest and say, 'This is out of the scope of my knowledge.' Always respond directly to the user's query without deviation.
Context: {context} '''

system_message_template = "You are a professional therapist, act like one., Here's the Question : {question}, Previous Context: {context}"

user_message_template = "User Query: {question} Answer:"

# Create message templates
system_message = SystemMessagePromptTemplate.from_template(system_message_template)
user_message = HumanMessagePromptTemplate.from_template(user_message_template)

# Compile messages into a chat prompt template
messages = [system_message, user_message]
chatbot_prompt = ChatPromptTemplate.from_messages(messages)

In [13]:
chatbot_prompt

ChatPromptTemplate(input_variables=['context', 'question'], messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], template="You are a professional therapist, act like one., Here's the Question : {question}, Previous Context: {context}")), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['question'], template='User Query: {question} Answer:'))])

In [14]:
# aka custom_template

condense_question_prompt = """Given the following conversation and a follow-up message, \
rephrase the follow-up message to a stand-alone question or instruction that \
represents the user's intent, add all context needed if necessary to generate a complete and \
unambiguous question or instruction, only based on the history, don't make up messages. \
Maintain the same language as the follow up input message.

Chat History:
{chat_history}

Follow Up Input: {question}
Standalone question or instruction:"""

### Generator

### Huggingface

In [15]:
# from langchain_community.llms.huggingface_pipeline import HuggingFacePipeline
# from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
# import torch

# model_id = r"C:\Users\User\Documents\aiml\LLM-TuningLab\MentalMate\models\decoder-only\Mistral-7B-Instruct-v0.2"
# tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
# model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16, trust_remote_code=True,low_cpu_mem_usage=True)

In [16]:
# model.to_bettertransformer()

In [17]:
# pipe = pipeline("text-generation", model=model, tokenizer=tokenizer, max_new_tokens=256)
# hf = HuggingFacePipeline(pipeline=pipe)

In [18]:
# text = "<s>[INST] What is your favourite condiment? [/INST]"
# "Well, I'm quite partial to a good squeeze of fresh lemon juice. It adds just the right amount of zesty flavour to whatever I'm cooking up in the kitchen!</s> "
# "[INST] Do you have mayonnaise recipes? [/INST]"
# hf(text)

In [19]:
# prompt = '''Human: Hello, who are you?
# AI: Greetings! I am an Mental Helth Therapist. How can I help you today?
# Human: I am Feeling Lazy.
# AI:I understand. Feeling Lazy can be a common emotion. Let's explore some strategies to help you overcome it.
# Human: Let Me About This Strategies.
# AI:'''
# hf(prompt)

### Ollama

In [20]:
# from langchain.llms import Ollama

In [21]:
# llm = Ollama(base_url="http://localhost:11434",model="llama2")

In [22]:
# llm("Hello")

### CTransformers

In [23]:
# Model used : https://huggingface.co/TheBloke/Llama-2-7B-Chat-GGUF
# Update with : https://huggingface.co/TheBloke/Llama-2-13B-chat-GGUF
# CTransformers config : https://github.com/marella/ctransformers#config

# config = {'max_new_tokens': 256,
#           'temperature': 0.4,
#           'repetition_penalty': 1.1,
#           'context_length': 4096, # Set to max for Chat Summary, Llama-2 has a max context length of 4096
#           }

# llm = CTransformers(model='W:\\Projects\\LangChain\\models\\quantizedGGUF-theBloke\\llama-2-7b-chat.Q2_K.gguf', 
#                     callbacks=[StreamingStdOutCallbackHandler()],
#                     config=config)

In [24]:
# from langchain_community.llms import CTransformers

In [25]:
# from ctransformers import AutoModelForCausalLM

In [26]:
# model_id = r"C:\Users\User\Documents\aiml\LLM-TuningLab\MentalMate\models\decoder-only\LlamaGuard-7B-GGUF"

# # Set gpu_layers to the number of layers to offload to GPU. Set to 0 if no GPU acceleration is available on your system.
# llm = AutoModelForCausalLM.from_pretrained(model_id, model_file="llamaguard-7b.Q3_K_M.gguf", model_type="llama", gpu_layers=50)

In [27]:
# print(llm("AI is going to"))

### Orchestrator

In [28]:
from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ChatMessageHistory,ConversationSummaryBufferMemory,ConversationBufferMemory

In [29]:
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

In [ ]:
# memory = ConversationSummaryBufferMemory(
#         memory_key="chat_history",
#         input_key="question",
#         llm=llm,
#         max_token_limit=40,
#         return_messages=True
#     )

In [30]:
import torch

In [31]:
# chain = ConversationalRetrievalChain.from_llm(llm = llm,
#                                               retriever=retriever,
#                                               memory = memory,
#                                               rephrase_question=False)

In [32]:
# torch.cuda.is_available()

In [33]:
# qa = ConversationalRetrievalChain.from_llm(
#     llm,
#     retriever=retriever,
#     memory = memory,
#     return_source_documents=False,
#     chain_type="stuff",
#     max_tokens_limit=100, # Llama-2 max = 4096
#     # condense_question_prompt= PromptTemplate.from_template(condense_question_prompt),
#     combine_docs_chain_kwargs={'prompt': chatbot_prompt},
#     verbose=True,
#     return_generated_question=False,
# )

### ChatBot

In [35]:
# chain.memory.buffer

In [ ]:
history = ChatMessageHistory()

In [ ]:
history

In [ ]:
# def ask(question: str):
#     answer = qa({"question": question,"chat_history":history.messages})["answer"]
#     print("##------##")
#     # print(answer)
#     return answer

# ask("I'm Tired all the time, feeling “lazy”")
# ask("I Think Its because of Social Media as I am a Socia Media Influencer.")

In [ ]:
question = "I'm Tired all the time, feeling lazy"
result = chain({"question": question, "chat_history": history.messages})

In [ ]:
result

In [ ]:
llm("I am Debanjan. Tell My Name")

In [ ]:
!pip install --upgrade --quiet  langchain-google-genai

In [ ]:
import getpass
import os

if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = "REDACTED_GOOGLE_API_KEY"

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(model="models/embedding-001")
vector = embeddings.embed_query("hello, world!")

In [ ]:
vector[:5]

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash-latest")

In [ ]:
result = llm.invoke("Write a ballad about LangChain")
print(result.content)